# Trading agent: assignment workflow with SEC and PPO

This notebook calls the repository modules; it does not maintain a second trading
implementation. The original assignment notebook remains the rubric reference.
Run GPU stages on a CUDA runtime or submit the matching `scripts/run_turing.slurm`
commands from Turing. **Run and review one stage at a time.** Training, provider
calls and paper orders have not been executed as part of this notebook's creation.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "project.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the llm_trading_agent repository")
PYTHON = ROOT / ".venv/bin/python"
if not PYTHON.is_file():
    PYTHON = Path(sys.executable)
config = json.loads((ROOT / "project.json").read_text())
RUN = (ROOT / config["run_dir"]).resolve()
print("Repository:", ROOT)
print("Source:", config["source"])
print("Run:", RUN)

def run(*args):
    command = [str(PYTHON), "-m", "agent", *args]
    if args[0] in {"collect", "sft", "ppo", "evaluate"}:
        command.append("--cuda")
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        status = process.wait()
    if status:
        raise RuntimeError(f"Stage failed with exit {status}; inspect its output and saved records")


## Tasks 1–2: tools, sources and decision context

Credentials are in the ignored `.env`, with the SEC contact value quoted. Do not
paste credentials into this notebook. `check` verifies the source, 100-symbol
universe, chronological splits and news/SEC coverage without loading weights.
The tool registry includes Alpaca news, price history, RSI, local MACD, quote,
SEC filing text and `search_market_news`. Live discovery omits Alpaca's symbols
filter; historical discovery uses only available archived whitelist news.


In [ ]:
run("check")


## Task 4a: actual base-model trajectories

The two-session smoke is a prefix of the fixed 60-session recent training fold,
20-session earlier-midterm fold and 20-session validation fold. Inspect actual
tool calls → result previews → raw final text → decisions → fills in each journal.
The full run resumes those plans and exports only valid profitable traded GOOD
episodes. The source archive is not regenerated.


In [ ]:
run("collect", "--smoke")


In [ ]:
for fold in ("train", "train-midterm", "validation"):
    paths = sorted((RUN / "collections" / fold).glob("episode-*.json"))
    print("\nFOLD", fold, "episodes", len(paths))
    for path in paths:
        row = json.loads(path.read_text())["record"]
        print(row["session"], row["policy_status"], row["label"], "fills", len(row["fills"]))
        for call in row["tool_calls"]:
            print("TOOL", call["tool"], "RESULT", json.dumps(call["result"])[:300])
        print("RAW FINAL", row["completion"][:1200])
        print("DECISIONS", row["decisions"])


In [ ]:
run("collect", "--resume")


## Task 4b: completion-only LoRA SFT

Preflight reports the actual GOOD corpus/token budget. The default optimizer
budget approximates three corpus passes, with measured trainable parameters
below 0.1%. Smoke stops after two steps of that same scheduler. Resume restores
optimizer, scheduler and RNG; it does not restart training from scratch.


In [ ]:
run("sft", "--preflight")


In [ ]:
run("sft", "--smoke")


In [ ]:
run("sft", "--resume")


## Optional extension: PPO

PPO starts from the completed SFT adapter and compiles train-only news/SEC episodes.
It trains final decisions in a simpler one-stock environment, not the full tool
loop. Review invalid fraction, reward, KL and update ratio at each pilot. Training
speed or positive optimizer reward does not establish trading performance.


In [ ]:
run("ppo", "--smoke")


In [ ]:
# Run after reviewing the two-update pilot:
# run("ppo", "--resume", "--updates", "10")
# Then, after another review:
# run("ppo", "--resume")


## Task 4c: matched validation comparison

Every available candidate uses the same chronological account-aware validation
plan and $100,000 starting cash. Invalid decisions and rejected orders remain in
metrics. Compare real decision/trade activity as well as returns. Do not select
using the separate October held-out week.


In [ ]:
run("evaluate", "--model", "all", "--smoke")


In [ ]:
run("evaluate", "--model", "all", "--resume")


In [ ]:
for name in ("base", "sft", "ppo"):
    path = RUN / f"eval-{name}" / "research_metrics.json"
    if path.is_file():
        metrics = json.loads(path.read_text())
        keys = ("sessions", "starting_cash", "ending_equity", "dollar_profit",
                "percentage_return", "executed_trades", "valid_schema_rate", "max_drawdown")
        print(name, {key: metrics.get(key) for key in keys})


## Tasks 3 and 5: BM25 reflections and the paper loop

Start with an isolated paper account and the selected model. This reads live
broker/data even in dry mode. Finite multi-hour loops are supported; exact
confirmed order state is reconciled independently from Qwen decisions. Closed or
uncertain broker state pauses; recoverable model/data failures do not invent trades.

Use a distinct stable session prefix for dry and submitting sessions. Reuse the
prefix after a disconnect to avoid duplicate trades. The model loads once per
session. Logs use ET dates; memory/outbox persist in `RUN/paper/MODEL`.

Current reflections use delayed confirmed **closed** lots, rather than delayed
portfolio marks. A buy-and-hold run may produce no new reflection; report that
limitation and show actual generated/retrieved lessons rather than claiming them.


In [ ]:
MODEL = "sft"  # choose once using validation; optionally "ppo"
PAPER_SUBMIT = False
SESSION = "rehearsal-001"  # use a DIFFERENT prefix when enabling paper submission
# After reviewing model/source/account readiness, explicitly run:
# run("paper", "--model", MODEL, "--cycle-id", SESSION, "--iterations", "10",
#     "--interval", "60", *(["--submit-paper"] if PAPER_SUBMIT else []))


## Report and submission evidence

Record base collection day counts, GOOD/BAD/invalid counts and sample labels;
LoRA parameter fraction and finite losses; base/SFT/PPO validation differences;
2–3 reasoned HOLD examples when present; actual tool, memory and reflection traces;
then ten consecutive live iterations, at least one Alpaca paper order/status,
and November 9–13 daily logs/order history/final balance. Keep model and settings
frozen during the live week. Historical October replay is a separate assessment.

Intentional differences from the starter: native Qwen chat templates; timestamp-safe
archived replay tools; next-open fills with costs; positive-net GOOD labels; exact
short evidence references; durable paper reconciliation. See `design.md`.
